# Pair Trading Backtest
Pulls from `pair_trading_engine.py`. All config is in the Configuration cell below — edit there and re-run.

In [1]:
import sys
from pathlib import Path

# Make sure engine is importable (adjust if notebook is not in project root)
sys.path.insert(0, str(Path(".").resolve()))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

from src.pair_trading_engine import PairTradingConfig, run, MultiPairTradingConfig

print("Engine loaded OK.")

Engine loaded OK.


## Configuration
Edit the hyperparameters and selected trade pairs in the configuration cell below.

In [2]:
base_pair_kwargs = dict(
    entry_z=2.0,
    exit_z=0.5,
    stop_loss_z=3.5,
    direction="long_short",
    ols_window=252,
    rebalance_freq_days=60,
    sizing="dollar_neutral",
    capital=1_000_000.0,
    fixed_notional=100_000.0,
    vol_window=20,
    commission_rate=0.0003,
    stamp_duty_rate=0.001,
)

cfg = PairTradingConfig(
    # ── Pair ─────────────────────────────────────────────────────────────────
    ts_code_a="002407.SZ",
    ts_code_b="002709.SZ",

    # ── Date range ───────────────────────────────────────────────────────────────
    start_date="20160101",
    end_date="20251231",

    **base_pair_kwargs,
)

# Example multi-pair trading configuration using the selected candidate pairs.
pair_cfgs = [
    PairTradingConfig(
        ts_code_a="600010.SH",
        ts_code_b="600111.SH",
        pair_name="pair_1",
        start_date="20220101",
        end_date="20231231",
        **base_pair_kwargs,
    ),
    PairTradingConfig(
        ts_code_a="600030.SH",
        ts_code_b="601211.SH",
        pair_name="pair_2",
        start_date="20220101",
        end_date="20231231",
        **base_pair_kwargs,
    ),
    PairTradingConfig(
        ts_code_a="000762.SZ",
        ts_code_b="600338.SH",
        pair_name="pair_3",
        start_date="20220101",
        end_date="20231231",
        **base_pair_kwargs,
    ),
    PairTradingConfig(
        ts_code_a="601788.SH",
        ts_code_b="601881.SH",
        pair_name="pair_4",
        start_date="20220101",
        end_date="20231231",
        **base_pair_kwargs,
    ),
    PairTradingConfig(
        ts_code_a="002407.SZ",
        ts_code_b="002709.SZ",
        pair_name="pair_5",
        start_date="20220101",
        end_date="20231231",
        **base_pair_kwargs,
    ),
]

multi_pair_cfg = MultiPairTradingConfig(
    pairs=pair_cfgs,
    start_date="20220101",
    end_date="20231231",
    capital=1_000_000.0,
    cash_buffer_pct=0.05,
    db_path=Path("data/prices.sqlite"),
)

cfg, pair_cfgs, multi_pair_cfg

(PairTradingConfig(ts_code_a='002407.SZ', ts_code_b='002709.SZ', pair_name=None, entry_z=2.0, exit_z=0.5, stop_loss_z=3.5, direction='long_short', ols_window=252, rebalance_freq_days=60, sizing='dollar_neutral', capital=1000000.0, fixed_notional=100000.0, vol_window=20, commission_rate=0.0003, stamp_duty_rate=0.001, start_date='20160101', end_date='20251231', model_start=None, model_end=None, test_start=None, test_end=None),
 [PairTradingConfig(ts_code_a='600010.SH', ts_code_b='600111.SH', pair_name='pair_1', entry_z=2.0, exit_z=0.5, stop_loss_z=3.5, direction='long_short', ols_window=252, rebalance_freq_days=60, sizing='dollar_neutral', capital=1000000.0, fixed_notional=100000.0, vol_window=20, commission_rate=0.0003, stamp_duty_rate=0.001, start_date='20220101', end_date='20231231', model_start=None, model_end=None, test_start=None, test_end=None),
  PairTradingConfig(ts_code_a='600030.SH', ts_code_b='601211.SH', pair_name='pair_2', entry_z=2.0, exit_z=0.5, stop_loss_z=3.5, direction

## Run Backtest

In [3]:
result = run(multi_pair_cfg)
print("Backtest complete.")
for w in result.warnings:
    print(" >", w)

Backtest complete.
 > pair_1: initial OLS on first 252 days: α=-0.4370, β=0.5123, R²=0.6876 | spread μ=-0.0000, σ=0.0631
 > pair_2: initial OLS on first 252 days: α=1.6996, β=1.0844, R²=0.7486 | spread μ=0.0000, σ=0.0412
 > pair_3: initial OLS on first 252 days: α=2.3525, β=0.8026, R²=0.6327 | spread μ=-0.0000, σ=0.0949
 > pair_4: initial OLS on first 252 days: α=1.6553, β=0.5124, R²=0.0647 | spread μ=0.0000, σ=0.1168
 > pair_5: initial OLS on first 252 days: α=0.3511, β=0.7771, R²=0.6168 | spread μ=0.0000, σ=0.0792


## Summary Statistics

In [4]:
pair_labels = result.stats.get("pair_labels", list(result.spread_dfs.keys()))
if not isinstance(pair_labels, list):
    pair_labels = [pair_labels]

stats_df = pd.DataFrame.from_dict(result.stats, orient="index", columns=["Value"])
stats_df.index.name = "Metric"

# Pretty labels
label_map = {
    "total_return_pct":          "Total Return (%)",
    "cagr_pct":                  "CAGR (%)",
    "sharpe_ratio":              "Sharpe Ratio (ann.)",
    "sortino_ratio":             "Sortino Ratio (ann.)",
    "max_drawdown_pct":          "Max Drawdown (%)",
    "calmar_ratio":              "Calmar Ratio",
    "n_round_trips":             "# Round-Trip Trades",
    "win_rate_pct":             "Win Rate (%)",
    "avg_round_trip_pnl_cny":    "Avg Round-Trip PnL (¥)",
    "total_transaction_costs_cny": "Total Transaction Costs (¥)",
    "start_value_cny":           "Starting Portfolio Value (¥)",
    "end_value_cny":             "Ending Portfolio Value (¥)",
    "rebalance_freq_days":       "Rebalance Frequency (days)",
    "pair_labels":               "Traded Pairs",
}
stats_df.index = [label_map.get(k, k) for k in stats_df.index]

# Format numeric values, leaving lists and strings intact.
def fmt_value(v):
    if isinstance(v, (int, np.integer)):
        return f"{v:,}"
    if isinstance(v, (float, np.floating)):
        return f"{v:,.4f}"
    return v

caption = (
    f"Backtest pairs: {', '.join(pair_labels)}  |  "
    f"Period: {result.config.start_date} → {result.config.end_date}"
)

display(stats_df.style
    .format(fmt_value)
    .set_table_styles([{
        "selector": "th",
        "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "13px")]
    }])
    .set_caption(caption))

if len(pair_labels) > 1:
    display(pd.DataFrame({"Selected pairs": pair_labels}))


,Value
Total Return (%),74.9400
CAGR (%),32.6000
Sharpe Ratio (ann.),2.1400
Sortino Ratio (ann.),2.6240
Max Drawdown (%),-12.1300
Calmar Ratio,2.6880
# Round-Trip Trades,62
Win Rate (%),48.3900
Avg Round-Trip PnL (¥),"2,028.2400"
Total Transaction Costs (¥),"71,970.5400"


,Selected pairs
0,pair_1
1,pair_2
2,pair_3
3,pair_4
4,pair_5


## Chart 1 — Equity Curve

In [5]:
eq = result.equity_curve

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    subplot_titles=["Portfolio Value (¥)", "Drawdown (%)"],
    vertical_spacing=0.08,
)

fig.add_trace(
    go.Scatter(x=eq.index, y=eq["portfolio_value"],
               name="Portfolio", line=dict(color="#4fc3f7", width=2)),
    row=1, col=1
)

# Drawdown
rolling_max = eq["portfolio_value"].cummax()
drawdown = (eq["portfolio_value"] - rolling_max) / rolling_max * 100
fig.add_trace(
    go.Scatter(x=eq.index, y=drawdown, name="Drawdown",
               fill="tozeroy", line=dict(color="#ef5350", width=1),
               fillcolor="rgba(239,83,80,0.2)"),
    row=2, col=1
)

# Mark trade entries
if not result.trade_log.empty:
    entries = result.trade_log[
        result.trade_log["action"].isin(["BUY", "SHORT_OPEN"]) &
        (result.trade_log["leg"] == "A")
    ].copy()
    entries["date"] = pd.to_datetime(entries["date"])
    entry_vals = eq["portfolio_value"].reindex(entries["date"], method="nearest")
    fig.add_trace(
        go.Scatter(
            x=entries["date"], y=entry_vals.values,
            mode="markers",
            marker=dict(symbol="triangle-up", size=10, color="#66bb6a"),
            name="Trade Entry"
        ),
        row=1, col=1
    )

fig.update_layout(
    height=600, template="plotly_dark",
    title=f"Equity Curve — {cfg.ts_code_a} / {cfg.ts_code_b}",
    legend=dict(orientation="h", y=1.02)
)
fig.show()

## Chart 2 — Spread & Z-score with Trade Signals

In [9]:
pair_labels = list(result.spread_dfs.keys())
if not pair_labels:
    raise ValueError("No pair spread data available for chart generation.")

pair_cfg_by_label = {
    pair_cfg.pair_name or f"{pair_cfg.ts_code_a}/{pair_cfg.ts_code_b}": pair_cfg
    for pair_cfg in pair_cfgs
}

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[0.35, 0.40, 0.25],
    subplot_titles=["Adjusted Close Prices (rebased to 100)", "Z-Score of Spread", "Rolling Hedge Ratio (Beta)"],
    vertical_spacing=0.07,
)

trace_pair_labels = []
for pair_label in pair_labels:
    sp = result.spread_dfs[pair_label]
    pair_cfg = pair_cfg_by_label.get(pair_label)
    visible = pair_label == pair_labels[0]

    base_a = sp["adj_close_a"].iloc[0]
    base_b = sp["adj_close_b"].iloc[0]

    fig.add_trace(
        go.Scatter(
            x=sp.index,
            y=sp["adj_close_a"] / base_a * 100,
            name=f"{pair_label}: A",
            line=dict(color="#4fc3f7"),
            visible=visible,
        ),
        row=1, col=1,
    )
    trace_pair_labels.append(pair_label)

    fig.add_trace(
        go.Scatter(
            x=sp.index,
            y=sp["adj_close_b"] / base_b * 100,
            name=f"{pair_label}: B",
            line=dict(color="#ffb74d"),
            visible=visible,
        ),
        row=1, col=1,
    )
    trace_pair_labels.append(pair_label)

    fig.add_trace(
        go.Scatter(
            x=sp.index,
            y=sp["z_score"],
            name="Z-Score",
            line=dict(color="#ce93d8", width=1.5),
            visible=visible,
        ),
        row=2, col=1,
    )
    trace_pair_labels.append(pair_label)

    if not result.trade_log.empty and "pair" in result.trade_log.columns:
        tlog = result.trade_log.copy()
        tlog["date"] = pd.to_datetime(tlog["date"])
        pair_trades = tlog[tlog["pair"] == pair_label]
        a_trades = pair_trades[pair_trades["leg"] == "A"].copy()
        a_trades = a_trades[a_trades["date"].isin(sp.index)]

        if not a_trades.empty:
            entries = a_trades[a_trades["action"].isin(["BUY", "SHORT_OPEN"])]
            exits = a_trades[a_trades["action"].isin(["SELL", "SHORT_CLOSE"])]

            if not entries.empty:
                fig.add_trace(
                    go.Scatter(
                        x=entries["date"],
                        y=sp.reindex(entries["date"])["z_score"],
                        mode="markers",
                        marker=dict(symbol="triangle-up", size=9, color="#66bb6a"),
                        name="Entries",
                        showlegend=False,
                        visible=visible,
                    ),
                    row=2, col=1,
                )
                trace_pair_labels.append(pair_label)

            if not exits.empty:
                fig.add_trace(
                    go.Scatter(
                        x=exits["date"],
                        y=sp.reindex(exits["date"])["z_score"],
                        mode="markers",
                        marker=dict(symbol="triangle-down", size=9, color="#ef5350"),
                        name="Exits",
                        showlegend=False,
                        visible=visible,
                    ),
                    row=2, col=1,
                )
                trace_pair_labels.append(pair_label)

    fig.add_trace(
        go.Scatter(
            x=sp.index,
            y=sp["hedge_ratio"],
            name="Beta",
            line=dict(color="#80cbc4", width=1.5),
            visible=visible,
        ),
        row=3, col=1,
    )
    trace_pair_labels.append(pair_label)

buttons = []
for pair_label in pair_labels:
    visible = [t == pair_label for t in trace_pair_labels]
    buttons.append({
        "label": pair_label,
        "method": "update",
        "args": [
            {"visible": visible},
            {"title": f"Spread Analysis — {pair_label}"},
        ],
    })

fig.update_layout(
    height=720,
    template="plotly_dark",
    title=f"Spread Analysis — {pair_labels[0]}",
    legend=dict(orientation="h", y=1.02),
    updatemenus=[{
        "active": 0,
        "buttons": buttons,
        "direction": "down",
        "x": 0.5,
        "y": 1.12,
        "xanchor": "center",
        "yanchor": "top",
    }],
    annotations=[{
        "text": "Select pair:",
        "x": 0.5,
        "y": 1.18,
        "xref": "paper",
        "yref": "paper",
        "showarrow": True,
        "arrowhead": 2,
        "ax": 0,
        "ay": 20,
        "font": {"color": "white"},
    }],
)
fig.show()


## Chart 3 — Trade Log

In [7]:
if result.trade_log.empty:
    print("No trades executed.")
else:
    tlog = result.trade_log.copy()
    tlog["date"] = pd.to_datetime(tlog["date"])

    # Colour-coded table
    action_colors = {
        "BUY":         "#c8e6c9",
        "SELL":        "#ffcdd2",
        "SHORT_OPEN":  "#fff9c4",
        "SHORT_CLOSE": "#e1bee7",
    }
    tlog["date_str"] = tlog["date"].dt.strftime("%Y-%m-%d")
    tlog["notional"] = tlog["notional"].round(0).astype(int)
    tlog["transaction_cost"] = tlog["transaction_cost"].round(2)

    display(tlog[["date_str", "leg", "ts_code", "action", "shares", "price",
                   "notional", "transaction_cost", "note"]]
            .rename(columns={"date_str": "date"})
            .style.map(
                lambda v: f"background-color: {action_colors.get(v, 'transparent')}",
                subset=["action"]
            )
            .set_caption(f"Full Trade Log ({len(tlog)} rows)"))

    # PnL bar chart per round trip
    entries = tlog[tlog["action"].isin(["BUY", "SHORT_OPEN"]) & (tlog["leg"] == "A")].reset_index(drop=True)
    exits   = tlog[tlog["action"].isin(["SELL", "SHORT_CLOSE"]) & (tlog["leg"] == "A")].reset_index(drop=True)
    n = min(len(entries), len(exits))

    if n > 0:
        pnls = []
        labels = []
        for j in range(n):
            e = entries.iloc[j]
            x = exits.iloc[j]
            # A-leg only PnL for illustration; full PnL is in portfolio curve
            if e["action"] == "BUY":
                p = x["notional"] - e["notional"] - e["transaction_cost"] - x["transaction_cost"]
            else:
                p = e["notional"] - x["notional"] - e["transaction_cost"] - x["transaction_cost"]
            pnls.append(p)
            labels.append(f"{e['date_str']} → {x['date_str']}")

        colors = ["#66bb6a" if p > 0 else "#ef5350" for p in pnls]
        fig2 = go.Figure(go.Bar(
            x=labels, y=pnls,
            marker_color=colors,
            text=[f"¥{p:,.0f}" for p in pnls],
            textposition="auto"
        ))
        fig2.update_layout(
            title="Round-Trip PnL per Trade (A-leg, gross)",
            template="plotly_dark",
            xaxis_tickangle=-45,
            yaxis_title="PnL (¥)",
            height=400,
        )
        fig2.show()

,date,leg,ts_code,action,shares,price,notional,transaction_cost,note
0,2022-01-07,A,600010.SH,SHORT_OPEN,18300,23.249070,425458,553.100000,
1,2022-01-07,B,600111.SH,BUY,500,827.190000,413595,124.080000,
2,2022-01-25,A,600010.SH,SHORT_CLOSE,18300,20.082530,367510,110.250000,mean_reversion
3,2022-01-25,B,600111.SH,SELL,500,793.905450,396953,516.040000,mean_reversion
4,2022-04-26,A,601788.SH,BUY,33900,13.112060,444499,133.350000,
5,2022-04-26,B,601881.SH,SHORT_OPEN,45300,9.808800,444339,577.640000,
6,2022-05-09,A,600010.SH,BUY,28900,15.332720,443116,132.930000,
7,2022-05-09,B,600111.SH,SHORT_OPEN,700,630.436950,441306,573.700000,
8,2022-05-25,A,002407.SZ,SHORT_OPEN,1900,224.063750,425721,553.440000,
9,2022-05-25,B,002709.SZ,BUY,800,536.722439,429378,128.810000,


## Export results

In [ ]:
pair_tag = f"{cfg.ts_code_a}_{cfg.ts_code_b}"
result.equity_curve.to_csv(f"res/equity_curve_{pair_tag}.csv")
result.spread_df.to_csv(f"res/spread_{pair_tag}.csv")
if not result.trade_log.empty:
    result.trade_log.to_csv(f"res/trade_log_{pair_tag}.csv", index=False)
pd.DataFrame([result.stats]).T.to_csv(f"res/stats_{pair_tag}.csv")
print("Exported equity curve, spread, trade log, and stats CSVs.")

Exported equity curve, spread, trade log, and stats CSVs.
